## 🏢 Agente Municipal — Listado y Gestión de Reportes

**Propósito:** Esta sección agrupa las operaciones y consultas correspondientes al rol de Agente Municipal. Incluye el flujo de trabajo para revisar la bandeja de entrada de reportes, filtrar por jurisdicción y actualizar el estado de los incidentes.

In [1]:
# Celda: inicializar 'mongo' usando MongoDAO (la misma conexión autenticada que usaste antes)
from dao.mongo_dao import MongoDAO

# Crear/recuperar wrapper autenticado (como en tu Celda 1)
mongo_dao = MongoDAO()
db = mongo_dao.connect()   # devuelve el objeto Database y deja mongo_dao.db/cliente listos si está implementado así

# Normalizar variable 'mongo' que esperan los DAOs (wrapper con .db y .client)
# Si MongoDAO ya expone .db y .client, adaptá; aquí hacemos un wrapper mínimo compatible.
class _MongoWrapper: pass
mongo = _MongoWrapper()
# Si MongoDAO expone client y db:
try:
    mongo.client = mongo_dao.client
    mongo.db = mongo_dao.db
except Exception:
    # Alternativa: si connect() devolvió la DB y mongo_dao guarda internamente el client
    try:
        mongo.db = db
        mongo.client = getattr(mongo_dao, "client", None)
    except Exception:
        # Fallback: si MongoDAO devuelve solo db, asignamos db y dejamos client None
        mongo.db = db
        mongo.client = None

print("Usando conexión MongoDAO. Base de datos:", getattr(mongo.db, "name", None))

# Verificación rápida de autenticación
try:
    info = mongo.client.admin.command("connectionStatus")
    print("connectionStatus:", info.get("authInfo", {}))
except Exception as e:
    print("No se pudo obtener connectionStatus (puede que mongo.client sea None o falten permisos):", type(e).__name__, e)


Usando conexión MongoDAO. Base de datos: civictech
connectionStatus: {'authenticatedUsers': [{'user': 'admin', 'db': 'admin'}], 'authenticatedUserRoles': [{'role': 'root', 'db': 'admin'}]}


In [2]:
# Celda: listar reportes por municipio/inspector automáticamente (ejecutar solo esta celda)
# Usa la conexión 'mongo' provista por MongoDAO (ya autenticada)
import importlib
import dao.agente_municipal_dao as amd
importlib.reload(amd)
from dao.agente_municipal_dao import AgenteMunicipalDAO
from bson import ObjectId
from datetime import datetime

def safe_str(v):
    try:
        if v is None:
            return ""
        if isinstance(v, datetime):
            return v.isoformat()
        return str(v)
    except Exception:
        return repr(v)

def trunc_text(s, length=80):
    s = safe_str(s)
    return s if len(s) <= length else s[:length-1] + "…"

def format_table(rows, headers):
    widths = []
    for i, h in enumerate(headers):
        col_vals = [str(r[i]) for r in rows] if rows else []
        max_cell = max([len(v) for v in col_vals], default=0)
        widths.append(max(len(h), max_cell))
    header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    sep = "-+-".join("-" * widths[i] for i in range(len(headers)))
    print(header_line)
    print(sep)
    for r in rows:
        print(" | ".join(str(r[i]).ljust(widths[i]) for i in range(len(headers))))
    print()

# 1) Obtener lista de municipio_id presentes en la colección 'reporte'
distinct_mids = list(mongo.db.reporte.distinct("municipio_id") or [])
if not distinct_mids:
    print("No se encontraron municipio_id en la colección 'reporte'.")
else:
    print(f"Municipios detectados en reportes: {len(distinct_mids)}\n")

# 2) Para cada municipio, resolver nombre y listar reportes usando AgenteMunicipalDAO
for mid in distinct_mids:
    # intentar obtener nombre del municipio (intentar buscar por ObjectId o por string)
    mun_doc = None
    try:
        mun_doc = mongo.db.municipio.find_one({"_id": mid}, {"nombre": 1})
    except Exception:
        # si la búsqueda por _id falló (p. ej. mid es string), intentar convertir a ObjectId
        try:
            mun_doc = mongo.db.municipio.find_one({"_id": ObjectId(str(mid))}, {"nombre": 1})
        except Exception:
            mun_doc = None

    mun_nombre = mun_doc.get("nombre") if mun_doc else safe_str(mid)

    # convertir mid a ObjectId para instanciar el DAO (AgenteMunicipalDAO exige ObjectId)
    try:
        agente_mid = ObjectId(str(mid))
    except Exception as e:
        print(f"⚠️ No se puede convertir municipio_id a ObjectId: {mid}  — salto este municipio.")
        continue

    # Instanciar DAO y listar reportes (la lógica de permisos está en el DAO)
    try:
        dao = AgenteMunicipalDAO(mongo, agente_municipio_id=agente_mid)
    except Exception as e:
        print(f"ERROR al instanciar AgenteMunicipalDAO para municipio {mun_nombre} ({agente_mid}): {type(e).__name__} {e}")
        continue

    # ensure_indexes puede fallar por permisos; no interrumpir
    try:
        dao.ensure_indexes()
    except Exception:
        pass

    try:
        reportes = dao.list_reports_by_municipio(limit=1000)
    except Exception as e:
        print(f"ERROR al listar reportes para municipio {mun_nombre} ({agente_mid}): {type(e).__name__} {e}")
        continue

    banner = "=" * 100
    print(banner)
    print(f"Municipio: {mun_nombre}  |  municipio_id: {agente_mid}  |  reportes: {len(reportes)}")
    print(banner + "\n")

    if not reportes:
        print("  (No hay reportes visibles para este municipio)\n")
        continue

    # preparar y mostrar tabla
    headers = ["N", "_id", "patente", "estado", "fechaHora_server", "usuario_id", "descripcion"]
    rows = []
    for i, r in enumerate(reportes, start=1):
        usuario_id = r.get("usuario_id")
        usuario_trunc = (str(usuario_id)[:8] + "…") if usuario_id else ""
        rows.append([
            f"{i:03d}",
            safe_str(r.get("_id")),
            trunc_text(r.get("patente_vehiculo"), 15),
            safe_str(r.get("estado")),
            safe_str(r.get("fechaHora_server")),
            usuario_trunc,
            trunc_text(r.get("descripcion"), 120)
        ])
    format_table(rows, headers)

    # resumen por estado
    estados_count = {}
    for r in reportes:
        e = r.get("estado") or "Sin Estado"
        estados_count[e] = estados_count.get(e, 0) + 1
    print("Resumen por estado:")
    for estado, cnt in sorted(estados_count.items(), key=lambda x: (-x[1], x[0])):
        print(f"  {estado}: {cnt}")
    print("\n")


Municipios detectados en reportes: 3

Municipio: Famatina  |  municipio_id: 6a28b11e3ca8ed0b349df8a3  |  reportes: 14

N   | _id                      | patente        | estado    | fechaHora_server           | usuario_id | descripcion                             
----+--------------------------+----------------+-----------+----------------------------+------------+-----------------------------------------
001 | 6a28ebfaac110dbb2e8577a1 | FAM-DEMO-01    | Pendiente | 2026-06-10T04:45:46.980000 | 6a28b11e…  | auto estacionado en la vereda           
002 | 6a28ebf8ac110dbb2e85779d | FAM-ELENA-02   | Pendiente | 2026-06-10T04:45:44.682000 | 6a28b11e…  | Auto abandonado                         
003 | 6a28ebf7ac110dbb2e85779b | FAM-VERA-01    | Pendiente | 2026-06-10T04:45:43.122000 | 6a28b11e…  | Auto estacionado en la vereda           
004 | 6a28ce5e8ae5cf8faac9a776 | FAM-NOEVID-001 | Pendiente | 2026-06-10T02:39:26.226000 | 6a28b11e…  | Prueba: intento insertar sin evidencia  
005 | 6a28c